# 30 — Gold: the model-ready feature table

**Layer:** Gold · **Engine:** PySpark · **Write format:** Delta

Gold is the **contract layer**. One table, one row per patient episode, human-readable
values, no train/test split, no imputation, no one-hot encoding. Everything downstream —
the training notebook, the clinician app, the Data Agent, a Power BI report — reads *this*
and nothing else.

### The line that matters: what does NOT belong in gold

| Belongs in gold | Belongs to the model (notebook 40) |
|---|---|
| Feature engineering that is pure business logic — comorbidity count, OKS subscales, provider attributes | Train/test split |
| The target label and its documented rule | MICE imputation |
| Readable categories (`"70 to 79"`, `"London"`) | One-hot encoding |

The reason is leakage. MICE imputation and the one-hot encoder are **fitted on training
data**. The moment you bake a train-fitted transform into a shared table, every consumer
of that table inherits information from the training fold — and you can no longer claim
your test metrics are honest. So those transforms live in the MLflow run, versioned with
the model, and are re-applied at serving time from the logged artifact.

This also keeps gold explainable: a clinician or a regulator can read a gold row.
They cannot read `age_band_4 = 1`.

## 1. Configuration

In [ ]:
LAKEHOUSE     = "NHS_PROMs_LH"
SILVER_TABLE  = "silver.knee_episode"
SILVER_DIM    = "silver.provider"
GOLD_TABLE    = "gold.knee_features"
GOLD_COHORT   = "gold.knee_preop_cohort"    # patients awaiting surgery (the app's inbox)

# The MCID threshold: an OKS improvement of more than 7 points is the smallest change a
# patient reliably notices. Below that, surgery did not deliver a meaningful benefit.
# Source: Beard et al., minimal clinically important difference for the Oxford Knee Score.
MCID_THRESHOLD = 7

spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")

from pyspark.sql import functions as F
print("ready")

## 2. Join the provider dimension

A left join, deliberately: a provider missing from the ODS registry must not silently
delete its patients from the cohort. Unmatched providers surface as `NULL` and get
imputed with everything else.

In [ ]:
silver   = spark.table(SILVER_TABLE)
provider = spark.table(SILVER_DIM)

df = silver.join(F.broadcast(provider), on="provider_code", how="left")

unmatched = df.filter(F.col("region").isNull()).select("provider_code").distinct().count()
print(f"Providers with no ODS match: {unmatched}  (left join — patients retained)")

## 3. Feature engineering

Three families of derived features, all **stateless** — every value depends only on its own
row. That is what makes them safe to compute before the train/test split: a transform that
never looks at another row cannot carry information across the split. The *fitted* transforms
(MICE, one-hot) stay in `40_train_register`, behind the split, where they belong.

* **`comorbidity_count`** — total burden across the twelve conditions. A clinician thinks in
  terms of "how sick is this patient overall", not twelve separate booleans.
* **OKS subscales** — the Oxford Knee Score's twelve items decompose into *pain*, *function*,
  and *activities of daily living*. The subscales are what a physiotherapist actually reasons
  about, and they make the EBM's shape functions readable on stage.
* **Sparse-category merges** — the 40–49 and 90–120 age bands are too thin to model alone.

### Two decisions that are not arithmetic

**Merged bands are labelled with their true span.** `40 to 49` and `50 to 59` become
`40 to 59`, not `50 to 59`. Labelling a 45-year-old as "50 to 59" would be a quiet lie in
every chart, every SHAP plot and every explanation shown to a clinician downstream.

**Suppressed demographics get an explicit `not_recorded` level — they are never imputed.**
NHS suppresses `age_band` and `sex` together when a (ageband, sex, provider, procedure) cell
holds 1–5 records, to protect confidentiality (see `31_data_exploration` §6 for the source).
Imputing them would mean reconstructing the value NHS deliberately hid, and — for sex, a
protected characteristic — serving clinical decisions on a guess. `not_recorded` is a real,
honest category: it says "this patient came from a small cell at their provider", which is
itself information the model may legitimately use.

Because gold leaves no NULLs in these two columns, MICE in notebook 40 has nothing to impute
there and skips them automatically. No special-casing needed.

In [ ]:
COMORBIDITY_COLS = [
    "heart_disease", "high_bp", "stroke", "circulation", "lung_disease", "diabetes",
    "kidney_disease", "nervous_system", "liver_disease", "cancer", "depression", "arthritis",
]

PAIN_ITEMS     = ["oks_t0_pain", "oks_t0_night_pain"]
FUNCTION_ITEMS = ["oks_t0_washing", "oks_t0_stairs", "oks_t0_walking",
                  "oks_t0_standing", "oks_t0_limping", "oks_t0_kneeling"]
ADL_ITEMS      = ["oks_t0_work", "oks_t0_confidence", "oks_t0_shopping", "oks_t0_transport"]

NOT_RECORDED = "not_recorded"

# Merged bins carry their ACTUAL span. Fixed a priori and hardcoded, which makes the
# mapping stateless -- see the markdown above.
AGE_BAND_GROUPS = {
    "40 to 49":  "40 to 59",
    "50 to 59":  "40 to 59",
    "60 to 69":  "60 to 69",
    "70 to 79":  "70 to 79",
    "80 to 89":  "80 to 120",
    "90 to 120": "80 to 120",
}

def _sum(cols):
    return sum((F.coalesce(F.col(c), F.lit(0)) for c in cols), F.lit(0))

age_expr = F.lit(NOT_RECORDED)                      # NULL (suppressed) falls through to here
for src, dst in AGE_BAND_GROUPS.items():
    age_expr = F.when(F.col("age_band") == src, F.lit(dst)).otherwise(age_expr)

# gender becomes a readable 3-level category. Gold is the human-readable layer, and
# "not_recorded" has to be visible as a value rather than hidden as a NULL.
gender_expr = (F.when(F.col("gender") == 1, F.lit("male"))
                .when(F.col("gender") == 0, F.lit("female"))
                .otherwise(F.lit(NOT_RECORDED)))

gold_df = (
    df
    .withColumn("comorbidity_count",      _sum(COMORBIDITY_COLS))
    .withColumn("oks_pain_subscale",      _sum(PAIN_ITEMS))
    .withColumn("oks_function_subscale",  _sum(FUNCTION_ITEMS))
    .withColumn("oks_adl_subscale",       _sum(ADL_ITEMS))
    .withColumn("age_band_grouped",       age_expr)
    .withColumn("sex",                    gender_expr)
    .withColumn("living_arrangements_grouped",
        F.when(F.col("t0_living_arrangements") == 3, F.lit(4))    # 84 rows -> merge into 4
         .otherwise(F.col("t0_living_arrangements")))
)

# Prove the merges did what they claim.
print("age_band -> age_band_grouped")
display(gold_df.groupBy("age_band", "age_band_grouped").count().orderBy("age_band"))
print("gender -> sex")
display(gold_df.groupBy("gender", "sex").count().orderBy("gender"))

# The suppression is a paired rule, so these two counts should match closely.
n_age = gold_df.filter(F.col("age_band_grouped") == NOT_RECORDED).count()
n_sex = gold_df.filter(F.col("sex") == NOT_RECORDED).count()
print(f"\nnot_recorded: age_band {n_age:,} | sex {n_sex:,} "
      f"(paired suppression -- expect these to be near-identical)")

display(gold_df.select("comorbidity_count", "oks_pain_subscale", "oks_function_subscale",
               "oks_adl_subscale").summary("min", "mean", "max"))

## 4. The target

`oks_delta = oks_t1_score - oks_t0_score`, and the label is **1 when the patient did NOT
improve meaningfully** (`delta <= 7`).

Note the polarity. The positive class is the *bad* outcome, because that is the class we
care about detecting — the patient for whom surgery will not deliver. It is also the
minority class, which is exactly why precision–recall, not accuracy, is the metric in
notebook 40.

`oks_t1_score` is a post-operative measurement. It is present in gold **only** to derive
the label, and notebook 40 drops every `t1_*` column before training. That guard is
asserted, not assumed.

In [ ]:
gold_df = (gold_df
    .withColumn("oks_delta", F.col("oks_t1_score") - F.col("oks_t0_score"))
    .withColumn("outcome_label",
        F.when(F.col("oks_delta") > MCID_THRESHOLD, F.lit(0))   # meaningful improvement
         .when(F.col("oks_delta").isNotNull(),      F.lit(1))   # poor outcome
         .otherwise(None))                                       # no follow-up: unlabelled
    .withColumn("has_label", F.col("outcome_label").isNotNull())
)

display(gold_df.groupBy("outcome_label").count().orderBy("outcome_label"))

labelled = gold_df.filter("has_label").count()
positives = gold_df.filter("outcome_label = 1").count()
print(f"Labelled episodes: {labelled:,}")
print(f"Poor-outcome rate: {positives / labelled * 100:.1f}%  <- the minority class")

## 5. Select the gold column set and write

Explicit column selection, not `select("*")`. A gold table that changes shape because
somebody added a column upstream is not a contract.

In [ ]:
GOLD_COLUMNS = [
    # ---- keys and lineage -------------------------------------------------
    "episode_id", "provider_code", "year",
    # ---- demographics -----------------------------------------------------
    "age_band", "age_band_grouped", "gender", "sex",
    # ---- pre-op status ----------------------------------------------------
    "t0_assisted", "t0_assisted_by", "t0_symptom_period", "t0_previous_surgery",
    "t0_living_arrangements", "living_arrangements_grouped", "t0_disability",
    # ---- EQ-5D dimensions -------------------------------------------------
    "t0_mobility", "t0_self_care", "t0_activity", "t0_discomfort", "t0_anxiety",
    # ---- comorbidities ----------------------------------------------------
    *COMORBIDITY_COLS, "comorbidity_count",
    # ---- Oxford Knee Score ------------------------------------------------
    *PAIN_ITEMS, *FUNCTION_ITEMS, *ADL_ITEMS, "oks_t0_score",
    "oks_pain_subscale", "oks_function_subscale", "oks_adl_subscale",
    # ---- provider context -------------------------------------------------
    "university_hospital", "independent_hospital", "region",
    # ---- target (post-op; excluded from features by notebook 40) ----------
    "oks_t1_score", "oks_delta", "outcome_label", "has_label",
    # ---- audit ------------------------------------------------------------
    "_batch_id", "_ingested_at_utc",
]

gold_out = (gold_df.select(*GOLD_COLUMNS)
                   .withColumn("_gold_built_at_utc", F.current_timestamp()))

(gold_out.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year")
    .saveAsTable(GOLD_TABLE))

spark.sql(f"OPTIMIZE {GOLD_TABLE} ZORDER BY (provider_code, episode_id)")
print(f"Wrote {GOLD_TABLE}: {gold_out.count():,} rows x {len(GOLD_COLUMNS)} columns")

## 6. Describe the table for humans and for the Data Agent

Column comments are not decoration. The Fabric Data Agent and Fabric IQ read this
metadata to ground natural-language questions — an undocumented column is a column the
agent will guess about. Ten minutes here buys you a demo that answers correctly.

In [ ]:
def _sql_str(text: str) -> str:
    """Escape a Python string for a Spark SQL single-quoted literal.

    Spark ends the literal at the first unescaped quote -- doubling it ('') does NOT
    escape, it terminates the string and starts a new one. Backslash is the escape.
    """
    return text.replace("\\", "\\\\").replace("'", "\\'")


COMMENTS = {
    "episode_id":            "Stable surrogate key for one patient surgical episode. Join key for the clinician app and risk scores.",
    "provider_code":         "NHS provider (hospital) code that performed the knee replacement.",
    "year":                  "NHS reporting year of the episode, e.g. 2018/19.",
    "age_band":              "Patient age band as supplied by NHS PROMs. NULL where suppressed for disclosure control.",
    "age_band_grouped":      "Age band, with the sparse 40-49 and 90-120 bands merged into 40-59 and 80-120. not_recorded means NHS suppressed the value for disclosure control (a 1-5 record cell), not that the patient declined to answer.",
    "gender":                "Raw NHS code: 1 = male, 0 = female, NULL = suppressed. Use `sex` for modelling and reporting.",
    "sex":                   "male, female, or not_recorded. not_recorded means NHS suppressed the value for disclosure control, never that it was imputed - this pipeline does not impute protected characteristics.",
    "oks_t0_score":          "Oxford Knee Score before surgery, 0-48. Lower is worse. The single strongest predictor.",
    "oks_pain_subscale":     "Pre-op OKS pain subscale (pain + night pain), 0-8. Lower is worse.",
    "oks_function_subscale": "Pre-op OKS function subscale (washing, stairs, walking, standing, limping, kneeling), 0-24.",
    "oks_adl_subscale":      "Pre-op OKS activities-of-daily-living subscale (work, confidence, shopping, transport), 0-16.",
    "comorbidity_count":     "Count of the 12 recorded long-term conditions. 0-12.",
    "t0_symptom_period":     "How long knee symptoms have been present. 1 = under 1 year, 2 = 1-5, 3 = 6-10, 4 = over 10.",
    "university_hospital":   "1 if the provider is an academic/teaching hospital.",
    "independent_hospital":  "1 if the provider is an independent-sector provider, 0 if an NHS trust.",
    "region":                "NHS England region of the provider.",
    "oks_t1_score":          "Oxford Knee Score six months after surgery. POST-OPERATIVE - never a model feature.",
    "oks_delta":             "oks_t1_score minus oks_t0_score. The improvement actually achieved.",
    "outcome_label":         "1 = poor outcome (OKS improvement of 7 points or less, below the minimal clinically important difference). 0 = meaningful improvement. This is the modelling target; the positive class is the poor outcome.",
    "has_label":             "TRUE when post-op follow-up exists. FALSE rows are unlabelled and excluded from training.",
}

spark.sql(f"COMMENT ON TABLE {GOLD_TABLE} IS "
          f"'Model-ready knee replacement episodes from NHS PROMs. One row per episode. "
          f"Human-readable: no imputation, no encoding, no train/test split. "
          f"Target = outcome_label (1 = poor outcome, OKS gain <= {MCID_THRESHOLD}).'")

for col, comment in COMMENTS.items():
    spark.sql(f"ALTER TABLE {GOLD_TABLE} CHANGE COLUMN {col} COMMENT '{_sql_str(comment)}'")

print(f"Documented {len(COMMENTS)} columns.")
display(spark.sql(f"DESCRIBE TABLE {GOLD_TABLE}"))

## 7. The pre-op cohort — the clinician app's inbox

The gold feature table is historical: every row already has a surgery date behind it.
The demo needs the opposite — patients *awaiting* surgery, with no outcome yet.

We build that cohort from the unlabelled episodes (real records, real feature values, no
follow-up questionnaire returned) and add the pseudonymous display identity the clinician
app shows. **Margaret is picked from here** — a real de-identified record, which is what
makes the on-stage narrative honest.

In [ ]:
from pyspark.sql import Window

# Deterministic pseudonyms so the demo is stable across re-runs and rehearsals.
FIRST_NAMES = ["Margaret", "Alan", "Priya", "David", "Joan", "Winston", "Eileen",
               "Raymond", "Doreen", "Colin", "Sylvia", "Nigel", "Beryl", "Patrick",
               "Maureen", "Trevor", "Gladys", "Dennis", "Hazel", "Roy"]
LAST_NAMES  = ["Hughes", "Okafor", "Sharma", "Whitfield", "Bennett", "Clarke", "Osei",
               "Fletcher", "Nowak", "Bailey", "Ahmed", "Sutcliffe", "Reid", "Kaur"]

cohort = (spark.table(GOLD_TABLE)
    .filter("has_label = false")
    .filter("oks_t0_score IS NOT NULL AND age_band IS NOT NULL"))

w = Window.orderBy("episode_id")
cohort = (cohort
    .withColumn("_n", F.row_number().over(w))
    .withColumn("display_name",
        F.concat_ws(" ",
            F.element_at(F.array(*[F.lit(n) for n in FIRST_NAMES]), (F.col("_n") % len(FIRST_NAMES)) + 1),
            F.element_at(F.array(*[F.lit(n) for n in LAST_NAMES]),  (F.col("_n") % len(LAST_NAMES))  + 1)))
    .withColumn("surgery_scheduled_date", F.date_add(F.current_date(), (F.col("_n") % 60) + 7))
    .drop("_n", "oks_t1_score", "oks_delta", "outcome_label", "has_label"))

(cohort.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(GOLD_COHORT))

spark.sql(f"COMMENT ON TABLE {GOLD_COHORT} IS "
          "'Pre-operative knee replacement patients awaiting surgery, with pseudonymous display names. "
          "The clinician app inbox. No outcome is known for these patients.'")

print(f"Wrote {GOLD_COHORT}: {cohort.count():,} pre-op patients")
display(cohort.select("episode_id", "display_name", "age_band_grouped", "sex", "oks_t0_score",
              "comorbidity_count", "provider_code", "surgery_scheduled_date").limit(10))

## 8. Pin Margaret

Run this once, note the `episode_id`, and hard-code it in the app and the rehearsal
script. Everything on stage — the endpoint call, the SHAP waterfall, the agent question —
points at this one id.

In [ ]:
margaret = (spark.table(GOLD_COHORT)
    .filter("display_name LIKE 'Margaret%'")
    .filter("age_band = '60 to 69' OR age_band = '70 to 79'")
    .filter("oks_t0_score BETWEEN 12 AND 22")      # genuinely symptomatic, plausibly borderline
    .orderBy("comorbidity_count", ascending=False)
    .limit(1))

display(margaret)

row = margaret.first()
if row:
    print(f"\nMARGARET_EPISODE_ID = \"{row['episode_id']}\"")
    print(f"  {row['display_name']}, {row['age_band']}, pre-op OKS {row['oks_t0_score']}, "
          f"{row['comorbidity_count']} comorbidities, provider {row['provider_code']}")
else:
    print("No match — relax the filters above.")

## 9. Verify lineage

Before you put the lineage graph on screen, prove it. This traces one gold row back to
the bronze batch and the supplier file it came from.

In [ ]:
display(spark.sql(f"""
    SELECT g.episode_id, g.provider_code, g.year, g.oks_t0_score,
           g._batch_id, g._ingested_at_utc, b._source_file
    FROM {GOLD_TABLE} g
    JOIN silver.knee_episode s ON s.episode_id = g.episode_id
    JOIN bronze.knee_provider b ON b._row_hash = s._row_hash
    LIMIT 5
"""))

print("Raw file -> bronze -> silver -> gold, joinable in one query. "
      "That is the record-keeping obligation, satisfied.")

---

**Next:** `40_train_register` — train on gold with MLflow tracking, and register the
calibrated EBM as a Fabric ML model.